# TP — Détection de beaconing C2

On dispose d'un dataframe contenant les logs de connexion sortantes sur un réseau d'entreprise.

Certains hôtes sont compromis, et font des appels réseau à un serveur « Commande et contrôle » (C2). Le but du TD est de
- identifier les hôtes compromis
- extraire les informations de compromission


## 0. Chargement des données

Le fichier `logs_reseau.csv` contient les connexions sortantes observées pendant
12 h (08:00 → 20:00) sur un réseau d'entreprise.

| colonne | description |
| --- | --- |
| `timestamp` | instant de la connexion |
| `src_ip` | hôte source (interne) |
| `dst_ip` | IP de destination |
| `domain` | domaine contacté (nom DNS) |
| `dst_port` | port de destination |
| `bytes_out` | octets envoyés |

In [1]:
import pandas as pd

df_logs_reseau = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
print(df_logs_reseau.shape)
df_logs_reseau.head()


(3211, 6)


,timestamp,src_ip,dst_ip,domain,dst_port,bytes_out
0,2026-10-14 08:00:05,10.0.0.66,185.165.29.10,sync-updates.example.net,443,512
1,2026-10-14 08:00:19,10.0.0.1,23.216.55.10,windowsupdate.com,443,120636
2,2026-10-14 08:00:22,10.0.0.67,193.29.164.30,flux-5741.xyz,443,256
3,2026-10-14 08:00:26,10.0.0.2,52.94.236.248,amazon.com,443,20114
4,2026-10-14 08:00:30,10.0.0.68,91.240.118.103,vpn-relay.xyz,443,128


In [2]:
df_logs_reseau.info()


<class 'pandas.DataFrame'>
RangeIndex: 3211 entries, 0 to 3210
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  3211 non-null   datetime64[us]
 1   src_ip     3211 non-null   str           
 2   dst_ip     3211 non-null   str           
 3   domain     3211 non-null   str           
 4   dst_port   3211 non-null   int64         
 5   bytes_out  3211 non-null   int64         
dtypes: datetime64[us](1), int64(2), str(3)
memory usage: 150.6 KB


## Exercice 1 — Beaconing « classique »

Un implant « beacon » rappelle son serveur C2 à intervalle **régulier**. Objectif :
repérer le couple (hôte, domaine) au rythme le plus régulier.

Pour cela, on va calculer le « coefficient de variation » (CV), sur les intervalle de temps entre requête.

   - `CV ≈ 0` → intervalle quasi constant (beacon *ou* tâche planifiée) ;
   - `CV` élevé → navigation humaine, irrégulière.

### Calcul du coefficient de variation


Étant donnée une liste **triée** de timestamp de requêtes, on peut calculer le coefficient de variation de la manière suivante:
- on commence par créer une nouvelle liste contenant les intervalles de temps, **en secondes**, entre chaque couple de requête successifs.
- sur cette liste, on applique la formule: `CV = écart-type / moyenne`

**Méthode, pas à pas :**

1. **Regrouper en listes** : fabriquer un DataFrame où chaque ligne est un couple
   `(src_ip, domain)` et où l'on range ses timestamps dans une **liste** (triée).
2. **Transformer chaque liste en écarts** : une petite fonction Python
   (`get_diffs`) qui renvoie les temps inter-arrivées, en secondes.
3. **Résumer en CV** : Calculer le coefficient de variation pour chacune de ces listes

In [3]:
def timestamps_par_src_dest(df: pd.DataFrame) -> pd.DataFrame:
    """
    À partir du dataframe logs_réseau, retourner un dataframe qui a en index
    (src_ip, domain), et une colonne `timestamps`, qui contient une liste triée 
    de tous les timestamps de connection associés à (src_ip, domain)
    """
    #BEGIN
    return (
        df.sort_values("timestamp")
        .groupby(["src_ip", "domain"])
        .agg(timestamps=("timestamp", list))
    )
    #END


In [4]:
ts_per_src_dest = timestamps_par_src_dest(df_logs_reseau)
nb_couples = df_logs_reseau[["src_ip", "domain"]].drop_duplicates().shape[0]
assert ts_per_src_dest.shape[0] == nb_couples
assert "timestamps" in ts_per_src_dest.columns
assert ts_per_src_dest["timestamps"].map(lambda ts: ts == sorted(ts)).all()
print("Étape 1 OK :", nb_couples, "couples (src_ip, domain) distincts")


Étape 1 OK : 590 couples (src_ip, domain) distincts


In [5]:
from datetime import datetime


def get_diffs(timestamps: list[datetime]) -> list[float]:
    """Temps (en secondes) entre instants successifs, dans l'ordre chronologique.

    La liste est triée au besoin : aucune hypothèse sur l'ordre en entrée.
    """
    #BEGIN
    tri = sorted(timestamps)
    res = []
    for i in range(len(tri) - 1):
        res.append((tri[i + 1] - tri[i]).total_seconds())
    return res
    #END


In [6]:
from datetime import datetime, timedelta

t0 = datetime(2026, 10, 14, 8, 0, 0)
assert get_diffs([t0, t0 + timedelta(seconds=60), t0 + timedelta(seconds=120)]) == [60.0, 60.0]
assert get_diffs([t0 + timedelta(seconds=120), t0, t0 + timedelta(seconds=60)]) == [60.0, 60.0]
assert get_diffs([t0]) == []
assert get_diffs([]) == []
print("get_diffs OK")


get_diffs OK


In [7]:
import statistics


def cv_destinations(df: pd.DataFrame, n_min: int = 20) -> pd.DataFrame:
    """Coefficient de variation des temps inter-arrivées par couple (src_ip, domain).

    Colonnes retournées : src_ip, domain, n, cv. On ignore les couples trop
    rares (moins de n_min connexions) : un CV sur 2 points ne veut rien dire.
    """
    #BEGIN
    # Le module `statistics` fournit `statistics.mean(liste)` et
    # `statistics.stdev(liste)` : moyenne et écart-type d'une liste de nombres.
    df["diffs"] = df["timestamps"].map(get_diffs)
    df["n"] = df["diffs"].map(len)
    df = df[df["n"] >= n_min]

    df["moyenne"] = df["diffs"].map(statistics.mean)
    df["ecart_type"] = df["diffs"].map(statistics.stdev)
    df["cv"] = df["ecart_type"] / df["moyenne"]
    return df.reset_index()[["src_ip", "domain", "n", "cv"]]
    #END


In [8]:
cv_destinations(ts_per_src_dest).sort_values("cv")

,src_ip,domain,n,cv
28,10.0.0.3,metrics-updates.example.net,23,0.001997
6,10.0.0.1,windowsupdate.com,23,0.002072
49,10.0.0.7,windowsupdate.com,23,0.002094
38,10.0.0.5,windowsupdate.com,23,0.002231
55,10.0.0.8,windowsupdate.com,23,0.002322
...,...,...,...,...
32,10.0.0.4,github.com,22,1.046264
3,10.0.0.1,github.com,23,1.098700
11,10.0.0.10,google.com,20,1.130925
5,10.0.0.1,microsoft.com,20,1.233611


### Liste blanche : les faux positifs périodiques

Le tri par CV ci-dessus montre que les couples les plus réguliers ne sont pas
forcément malveillants : les mises à jour (Windows, antivirus, NTP…) contactent
elles aussi leur serveur à intervalle quasi constant — avec un petit jitter réseau
du même ordre que celui du beacon. Ce sont des **faux positifs**.

L’équipe SOC nous fournit la liste des domaines légitimes connus (liste blanche) :
on les écartera avant de chercher le beacon. On **réutilisera cette liste dans la
suite du TD** chaque fois qu’il faudra écarter le trafic légitime.

Domaines whitelistés : `windowsupdate.com`, `ntp.org`, `av-updates.example.net`,
`sync-backup.example.net`, `metrics-updates.example.net`.

In [9]:
# Domaines légitimes périodiques (liste blanche fournie par le SOC).
# À réutiliser chaque fois qu’il faut écarter le trafic légitime.
DOMAINES_WHITELIST = {
    "windowsupdate.com",
    "ntp.org",
    "av-updates.example.net",
    "sync-backup.example.net",
    "metrics-updates.example.net",
}

In [10]:
def detecter_beacon_timing(df: pd.DataFrame) -> tuple[str, str]:
    """Renvoie (src_ip, domain) du beacon le plus régulier, hors liste blanche."""
    #BEGIN
    stats = cv_destinations(timestamps_par_src_dest(df))
    # Certains domaines légitimes (mises à jour, NTP…) sont *aussi* périodiques :
    # on les écarte avant de chercher le CV minimal.
    hors_whitelist = stats[~stats["domain"].isin(DOMAINES_WHITELIST)]
    suspect = hors_whitelist.loc[hors_whitelist["cv"].idxmin()]
    return suspect["src_ip"], suspect["domain"]
    #END


In [22]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
assert (src, domaine) == ("10.0.0.66", "sync-updates.example.net"), f"trouvé : {src} -> {domaine}"
print("ex 1 OK :", src, "->", domaine)

ex 1 OK : 10.0.0.66 -> sync-updates.example.net


### Extraction des IOC

Le couple (hôte, domaine) suspect étant identifié, on extrait ses **IOC**
(indicateurs de compromission) : les IP de destination distinctes
contactées par cet hôte pour ce domaine.

**Consigne :** compléter `extraire_ioc_beacon`, qui renvoie un dataframe
des destinations distinctes (une ligne par IP de destination), trié par `dst_ip`.


In [ ]:
def extraire_ioc_beacon(df: pd.DataFrame, src_ip: str, domain: str) -> pd.DataFrame:
    """IOC du beacon : destinations distinctes contactées par (src_ip, domain).

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par dst_ip.
    """
    #BEGIN
    return (
        df[(df["src_ip"] == src_ip) & (df["domain"] == domain)][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    #END


In [ ]:
ioc_beacon = extraire_ioc_beacon(df_logs_reseau, src, domaine)
assert len(ioc_beacon) == 1
assert ioc_beacon.iloc[0]["dst_ip"] == "185.165.29.10"
assert ioc_beacon.iloc[0]["domain"] == "sync-updates.example.net"
print("ioc ex 1 OK :", len(ioc_beacon), "IOC")


## Exercice 2 — Timing aléatoire : il ne reste que la plage d'IP

L'implant le plus sophistiqué **randomise totalement** son timing : plus aucun
rythme à détecter. Mais il doit toujours joindre son serveur C2, hébergé dans un
`/24` que l'entreprise ne contacte jamais.

**Méthode :**

1. Écarter le domaine du beacon déjà identifié à l'exercice 1
   (`sync-updates.example.net`).
2. Réduire chaque IP restante à son préfixe `/24`.
3. Pour chaque préfixe, calculer **deux chiffres séparés** :
   - `nb_sources` : combien d'hôtes distincts le touchent ;
   - `volume` : combien de connexions il reçoit.
4. Un préfixe **rare** (1 seule source) **et massivement contacté** = suspect.

**Note** Le timing aléatoire ne laisse aucun rythme exploitable :
ni la régularité (exercice 1) ni le comptage de destinations (exercice 3) ne le détectent.
Seule la plage d'IP le trahit.


In [15]:
def prefixe(ip: str) -> str:
    """Préfixe /24 d'une IPv4."""
    #BEGIN
    return ".".join(ip.split(".")[:3])
    #END

In [16]:
# DELETE
df_logs_reseau["prefix"] = df_logs_reseau["dst_ip"].map(prefixe)
df_logs_reseau

,timestamp,src_ip,dst_ip,domain,dst_port,bytes_out,prefix
0,2026-10-14 08:00:05,10.0.0.66,185.165.29.10,sync-updates.example.net,443,512,185.165.29
1,2026-10-14 08:00:19,10.0.0.1,23.216.55.10,windowsupdate.com,443,120636,23.216.55
2,2026-10-14 08:00:22,10.0.0.67,193.29.164.30,flux-5741.xyz,443,256,193.29.164
3,2026-10-14 08:00:26,10.0.0.2,52.94.236.248,amazon.com,443,20114,52.94.236
4,2026-10-14 08:00:30,10.0.0.68,91.240.118.103,vpn-relay.xyz,443,128,91.240.118
...,...,...,...,...,...,...,...
3206,2026-10-14 19:59:23,10.0.0.12,45.63.10.22,metrics-updates.example.net,443,19629,45.63.10
3207,2026-10-14 19:59:26,10.0.0.9,140.82.112.3,github.com,443,10301,140.82.112
3208,2026-10-14 19:59:36,10.0.0.68,91.240.118.101,vpn-relay.xyz,443,128,91.240.118
3209,2026-10-14 19:59:47,10.0.0.1,142.250.75.14,google.com,443,28449,142.250.75


# 2.1 Construction du dataframe et filtrage
Ajouter une colonne `prefix` au dataframe `df_logs_reseau`, puis obtenir le dataframe qui, pour chaque prefixe, 
donne le nombre de sources contactées, ainsi que le nombre total de requêtes.

Filtrer ensuite le dataframe pour enlever les lignes peu intéressantes.

In [17]:
def hotes_plage_rare(df: pd.DataFrame, vol_min: int = 50) -> list[str]:
    """Hôtes source touchant un /24 rare (1 source) ET massivement contacté."""
    #BEGIN
    ## Le beacon de l'exercice 1 est lui aussi sur un /24 rare : on exclut
    ## son domaine, déjà identifié, pour ne garder que le nouvel implant.
    df = df[~df["domain"].isin({"sync-updates.example.net"})].copy()
    df["prefix"] = df["dst_ip"].map(prefixe)

    # deux signaux, calculés séparément :
    nb_sources = df.groupby("prefix")["src_ip"].nunique()  # combien d'hôtes distincts ?
    volume = df.groupby("prefix")["timestamp"].count()      # combien de connexions ?

    rares = nb_sources[(nb_sources == 1) & (volume >= vol_min)].index
    return sorted(df[df["prefix"].isin(rares)]["src_ip"].unique())
    #END

In [18]:
suspects = hotes_plage_rare(df_logs_reseau)
assert suspects == ["10.0.0.68"], f"trouvé : {suspects}"
print("ex 2 OK :", suspects)


ex 2 OK : ['10.0.0.68']


### Extraction des IOC

**Consigne :** compléter `extraire_ioc_plage`, qui renvoie les destinations
distinctes contactées par l'hôte suspect (une ligne par IP de destination),
triées par `dst_ip`.


In [ ]:
def extraire_ioc_plage(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par dst_ip.
    """
    #BEGIN
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    #END


In [ ]:
ioc_plage = extraire_ioc_plage(df_logs_reseau, suspects[0])
assert len(ioc_plage) == 5
assert ioc_plage["dst_ip"].is_unique
assert (ioc_plage["dst_ip"].str.startswith("91.240.118.")).all()
print("ioc ex 2 OK :", len(ioc_plage), "IOC")


## Exercice 3 — Fast-flux : la destination qui tourne

Certains C2 changent d'IP **et** de domaine à chaque connexion pour échapper aux
filtres : c'est le **fast-flux** (ou DGA). Chaque domaine n'apparaissant qu'une
fois, grouper par domaine est inutile.

**Méthode :** regarder l'hôte *source*. Un hôte légitime contacte un petit nombre
de destinations. Un hôte en fast-flux contacte un grand nombre d'IP **distinctes**.


In [12]:
def detecter_fastflux(df: pd.DataFrame) -> str:
    """Renvoie l'hôte source qui contacte le plus d'IP de destination distinctes."""
    #BEGIN
    return df.groupby("src_ip")["dst_ip"].nunique().idxmax()
    #END


In [13]:
def extraire_ioc(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par dst_ip.
    """
    #BEGIN
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    #END


In [14]:
hote = detecter_fastflux(df_logs_reseau)
assert hote == "10.0.0.67", f"trouvé : {hote}"
print("ex 3 OK :", hote)

ioc = extraire_ioc(df_logs_reseau, hote)
assert len(ioc) == 40
assert ioc["dst_ip"].is_unique
print("  ->", len(ioc), "IOC (IP de destination distinctes)")


ex 3 OK : 10.0.0.67
  -> 40 IOC (IP de destination distinctes)


## Exercice 4 — Quand le comptage ne suffit plus

Le réseau grossit : on observe désormais une centaine d'hôtes, qui contactent
chacun des centaines d'IP de destination distinctes (CDN, edge, caches…).

Un hôte compromis renouvelle ses destinations à chaque connexion. Retrouvez-le.


In [19]:
df_logs_reseau_grand = pd.read_csv("logs_reseau_grand.csv", parse_dates=["timestamp"])
print(df_logs_reseau_grand.shape)
df_logs_reseau_grand.head()


(45449, 6)


,timestamp,src_ip,dst_ip,domain,dst_port,bytes_out
0,2026-10-14 07:59:58,10.0.0.43,198.51.100.10,telemetry.analytics.example.net,443,1535
1,2026-10-14 08:00:00,10.0.0.5,198.18.2.43,cdn-555.example.net,443,53243
2,2026-10-14 08:00:00,10.0.0.90,198.18.1.213,cdn-469.example.net,443,23296
3,2026-10-14 08:00:02,10.0.0.97,198.18.0.3,cdn-3.example.net,443,16395
4,2026-10-14 08:00:06,10.0.0.83,198.18.2.2,cdn-514.example.net,443,44607


**Consigne :** compléter `detecter_hote_compromis`, qui renvoie l'hôte compromis
du grand réseau.


In [20]:
def detecter_hote_compromis(df: pd.DataFrame) -> str:
    """Renvoie l'hôte compromis du grand réseau."""
    #BEGIN
    nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
    ip_rares = nb_sources[nb_sources == 1].index
    rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
    return rare_par_hote.idxmax()
    #END


In [21]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
assert victime == "10.0.0.167", f"trouvé : {victime}"
print("ex 4 OK :", victime)


ex 4 OK : 10.0.0.167


### Extraction des IOC

**Consigne :** compléter `extraire_ioc_grand`, qui renvoie les destinations
distinctes contactées par l'hôte compromis (une ligne par IP de destination),
triées par `dst_ip`.


In [ ]:
def extraire_ioc_grand(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par dst_ip.
    """
    #BEGIN
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    #END


In [ ]:
ioc_grand = extraire_ioc_grand(df_logs_reseau_grand, victime)
assert len(ioc_grand) == 400
assert ioc_grand["dst_ip"].is_unique
print("ioc ex 4 OK :", len(ioc_grand), "IOC")


## Bilan

| exercice | signal | notion pandas | hôte trouvé |
| --- | --- | --- | --- |
| 1 | régularité temporelle | `groupby` + `agg(list)` + CV | `10.0.0.66` |
| 2 | plage d'IP rare et concentrée | `map` + deux `groupby` | `10.0.0.68` |
| 3 | multiplicité des destinations | `groupby().nunique()` | `10.0.0.67` |
| 4 | destinations rares (une seule source) | `nunique` + `isin` | `10.0.0.167` |

**À retenir :** la détection de C2 est **heuristique** — on croise plusieurs
signaux (rythme, destinations, plage d'IP, rareté des destinations). Aucun
signal seul ne suffit, et chaque contre-mesure de l'attaquant (jitter,
fast-flux, timing aléatoire, destinations partagées) casse un signal… mais en
laisse un autre.
